# LLM segmentation
Inspect source blocks, request headings once, and review them in a DataFrame. Complete lists remain intact.

Run `uv sync` and select the project kernel. Set `run_live=True` to make a real model request using server environment credentials.

In [ ]:
from utils import setup_notebook

import pandas as pd
from IPython.display import Markdown, display
from data_platform.digest.segmenter.llm_e2e import (
    split_blocks,
    generate_block_headings,
    reconstruct_document,
)

project_root = setup_notebook()


## 1. Configure and preview

In [ ]:
input_path = project_root / ".data/article-en-01.md"
model = "openai:gpt-6-astra"
run_live = False
markdown_doc = input_path.read_text(encoding="utf-8")
blocks = split_blocks(markdown_doc)
blocks_df = pd.DataFrame({"block": range(len(blocks)), "text": blocks})
display(blocks_df)


## 2. Generate headings
The staged API makes one heading request; there is no second demonstration call.

In [ ]:
headings = None
segmented_doc = None
if run_live:
    headings = await generate_block_headings(blocks, model=model)
    segmented_doc = reconstruct_document(blocks, headings)

headings_df = blocks_df.assign(heading=headings if headings is not None else pd.NA)
display(headings_df)
if segmented_doc is not None:
    display(Markdown(segmented_doc))


## 3. Export (optional)
Saving replaces the generated Markdown and heading CSV beside the input file.

In [ ]:
save_output = False
if save_output and segmented_doc is not None:
    output_path = input_path.with_name(f"{input_path.stem}_llm_segmented.md")
    output_path.write_text(segmented_doc, encoding="utf-8")
    headings_df.to_csv(output_path.with_suffix(".csv"), index=False)
